## Widgets Configuration

In [0]:
# Databricks notebook source

# 00 — Setup Infrastructure
# Creates the Unity Catalog catalog, the five layer schemas, and the three
# raw-layer volumes used by every later notebook in this project.
# MAGIC
# Rerun-safe: every statement is `IF NOT EXISTS`.

# COMMAND ----------

# ## Widgets
# Values are supplied by DABs `base_parameters` in `databricks.yml` at job run
# time, and default here for interactive/manual runs.

# COMMAND ----------

dbutils.widgets.text("catalog_name", "vstone_catalog", "1. Catalog Name")
dbutils.widgets.text("raw_schema", "raw", "2. Raw Schema")
dbutils.widgets.text("bronze_schema", "bronze", "3. Bronze Schema")
dbutils.widgets.text("silver_schema", "silver", "4. Silver Schema")
dbutils.widgets.text("gold_schema", "gold", "5. Gold Schema")
dbutils.widgets.text("security_schema", "security", "6. Security Schema")

CATALOG = dbutils.widgets.get("catalog_name")
RAW_SCHEMA = dbutils.widgets.get("raw_schema")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")
SECURITY_SCHEMA = dbutils.widgets.get("security_schema")

## Catalog, Schemas, Volumes

In [0]:
ALL_SCHEMAS = [RAW_SCHEMA, BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA, SECURITY_SCHEMA]
RAW_VOLUMES = ["landing", "chunks", "checkpoints"]

print(f"INFO: Initializing catalog '{CATALOG}' with schemas {ALL_SCHEMAS}")

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"USE CATALOG {CATALOG}")
print(f"SUCCESS: Catalog '{CATALOG}' ready.")

for schema in ALL_SCHEMAS:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")
    print(f"SUCCESS: Schema '{CATALOG}.{schema}' ready.")

for volume in RAW_VOLUMES:
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{RAW_SCHEMA}.{volume}")
    print(f"SUCCESS: Volume '{CATALOG}.{RAW_SCHEMA}.{volume}' ready.")

## Verification

In [0]:
created_schemas = [r["databaseName"] for r in spark.sql(f"SHOW SCHEMAS IN {CATALOG}").collect()]
missing = [s for s in ALL_SCHEMAS if s not in created_schemas]

if missing:
    raise Exception(f"Setup incomplete — missing schemas: {missing}")

print(f"\n{'='*55}")
print(f"  INFRASTRUCTURE READY")
print(f"{'='*55}")
print(f"  Catalog  : {CATALOG}")
print(f"  Schemas  : {', '.join(ALL_SCHEMAS)}")
print(f"  Volumes  : {', '.join(f'{RAW_SCHEMA}.{v}' for v in RAW_VOLUMES)}")
print(f"{'='*55}")
print(f"\nNext step: upload cars.csv, telegram.csv, node_locations.csv,")
print(f"streets_list.csv to /Volumes/{CATALOG}/{RAW_SCHEMA}/landing/")
print(f"then run 01_data_profiling.")